Download prices from Yahoo Finance

In [6]:
# Import necessary packages

import os
import pandas as pd
import yfinance as yf

In [37]:
# Define ETF tickers, benchmark and macro data

UNIVERSE = ["SPY", "QQQ", "IWM", "EFA", "EEM", "TLT", "IEF", "SHY", "TIP", "LQD",
           "HYG", "EMB", "EMLC", "GLD", "DBC", "XLE", "VNQ", "UUP", "FXY", "FXA", "FXF"]

BENCHMARK = ["AGG"]
RISK_FREE = ["^IRX"]

# Use yfinance libray to download them into a dataframe
    # threads set to false makes sure tickers are downloaded in sequence and not in parallel (less likely to be blocked by yahoo finance)
    # progress set to false hides the progress bar 

raw_data = yf.download(UNIVERSE + BENCHMARK + RISK_FREE, start="2003-01-01", auto_adjust=True, group_by="column", threads=False, progress=False)

In [38]:
raw_data.info()

<class 'pandas.DataFrame'>
DatetimeIndex: 5989 entries, 2003-01-02 to 2026-10-21
Columns: 115 entries, ('Close', 'AGG') to ('Volume', '^IRX')
dtypes: float64(106), int64(9)
memory usage: 5.3 MB


Data exploration

In [41]:
# select only closing prices
universe_close  = raw_data["Close"][UNIVERSE]
benchmark_close = raw_data["Close"][BENCHMARK]
irx             = raw_data["Close"]["^IRX"]


# check for data types
universe_close.dtypes

Ticker
SPY     float64
QQQ     float64
IWM     float64
EFA     float64
EEM     float64
TLT     float64
IEF     float64
SHY     float64
TIP     float64
LQD     float64
HYG     float64
EMB     float64
EMLC    float64
GLD     float64
DBC     float64
XLE     float64
VNQ     float64
UUP     float64
FXY     float64
FXA     float64
FXF     float64
dtype: object

In [42]:
# check date range and type
universe_close.index.min(), universe_close.index.max()

(Timestamp('2003-01-02 00:00:00'), Timestamp('2026-10-21 00:00:00'))

In [72]:
# check for missing data for all tickers
universe_close.isna().sum().sort_values()

Ticker
SPY        0
QQQ        0
IWM        0
EFA        0
TLT        0
IEF        0
SHY        0
LQD        0
XLE        0
EEM       70
TIP      234
VNQ      438
GLD      474
DBC      779
FXA      876
FXF      876
FXY     1035
UUP     1046
HYG     1074
EMB     1250
EMLC    1902
dtype: int64

In [44]:
# check first date for each ticker

universe_close.apply(lambda s: s.first_valid_index()).sort_values()

Ticker
SPY    2003-01-02
QQQ    2003-01-02
IWM    2003-01-02
EFA    2003-01-02
TLT    2003-01-02
IEF    2003-01-02
SHY    2003-01-02
LQD    2003-01-02
XLE    2003-01-02
EEM    2003-04-14
TIP    2003-12-05
VNQ    2004-09-29
GLD    2004-11-18
DBC    2006-02-06
FXF    2006-06-26
FXA    2006-06-26
FXY    2007-02-13
UUP    2007-03-01
HYG    2007-04-11
EMB    2007-12-19
EMLC   2010-07-23
dtype: datetime64[s]

In [45]:
# define a function to locate gaps after fund inception 

def gaps_after_start (s):
    s = s.loc[s.first_valid_index():]
    return s.isna().sum()

In [46]:
universe_close.apply(gaps_after_start)

Ticker
SPY     0
QQQ     0
IWM     0
EFA     0
EEM     0
TLT     0
IEF     0
SHY     0
TIP     0
LQD     0
HYG     0
EMB     0
EMLC    0
GLD     0
DBC     0
XLE     0
VNQ     0
UUP     0
FXY     0
FXA     0
FXF     0
dtype: int64

In [48]:
# check to see if we have the correct ammount of trading days 

universe_close.groupby(universe_close.index.year).size()

Date
2003    252
2004    252
2005    252
2006    251
2007    251
2008    253
2009    252
2010    252
2011    252
2012    250
2013    252
2014    252
2015    252
2016    252
2017    251
2018    251
2019    252
2020    253
2021    252
2022    251
2023    250
2024    252
2025    250
2026    202
dtype: int64

In [52]:
# final check to see if there are a lot of 0 return day whcih could suggest data filling at the source

returns = universe_close.pct_change(fill_method = None)
(returns == 0).sum()

Ticker
SPY      21
QQQ      29
IWM      14
EFA      46
EEM      40
TLT      27
IEF      55
SHY     498
TIP      88
LQD      67
HYG      75
EMB      48
EMLC    103
GLD      20
DBC      85
XLE      35
VNQ      29
UUP     219
FXY      67
FXA      52
FXF      48
dtype: int64

In [54]:
# and for days where every ticker shows 0, indicating duplicate days

(returns == 0).all(axis=1).sum()

np.int64(0)

In [60]:
# 0 days return are relatively negligible, except for a few items. There seems to be no duplicated days
# We should check consecutive streaks of 0s. 5 or more in a row could be a problem

# we do this through a function

def max_zero_run(r):
    is_zero = (r == 0)
    run_id = (~is_zero).cumsum() # is_zero will be true for days when return did not move so we add ~ ;cumsum() will add the Trues generating IDs
    return is_zero.groupby(run_id).sum().max() # counts the 0 days with each ID and gives the longes streak with .max()

In [61]:
returns.apply(max_zero_run).sort_values(ascending=False)

Ticker
SHY     4
HYG     3
UUP     3
EEM     2
QQQ     2
FXA     2
FXF     2
EFA     2
TIP     2
EMLC    2
FXY     2
DBC     2
IEF     2
XLE     2
SPY     1
IWM     1
TLT     1
EMB     1
LQD     1
VNQ     1
GLD     1
dtype: int64

In [63]:
# let's check when SHY had 0 return days

shy_zero = (returns["SHY"] == 0)
shy_zero.groupby(shy_zero.index.year).sum()

Date
2003     7
2004    12
2005    20
2006    26
2007    14
2008     3
2009    12
2010    10
2011    28
2012    51
2013    56
2014    25
2015    23
2016    19
2017    21
2018    33
2019    13
2020    41
2021    39
2022     8
2023     5
2024     9
2025    11
2026    12
Name: SHY, dtype: int64

In [67]:
# SHY seems explanable by macro factors but we should also take a look at HYG and UUP's streaks

# We do this through a function which will print the dates asociated with max_zero_run function
def longest_zero_run_dates(r):
    is_zero = r == 0
    run_id = (~is_zero).cumsum()
    runs = is_zero.groupby(run_id).sum()
    longest = runs.idxmax()
    return r[(run_id == longest) & is_zero].index

for t in ["HYG", "UUP"]:
    print(t, list(longest_zero_run_dates(returns[t]).date))

HYG [datetime.date(2026, 1, 13), datetime.date(2026, 1, 14), datetime.date(2026, 1, 15)]
UUP [datetime.date(2012, 12, 24), datetime.date(2012, 12, 26), datetime.date(2012, 12, 27)]


In [73]:
# UUP does not seem to be a problem as the dates revolve around xmas period 

# for HYG instead, we need to check. is a large and heavily traded ETF
# we do this by selecting certain columns and observing january prices
cols = ["Open", "High", "Low", "Close", "Volume"]
raw_data.loc["2026-01-08":"2026-01-21", [(c, "HYG") for c in cols]]

Price,Open,High,Low,Close,Volume
Ticker,HYG,HYG,HYG,HYG,HYG
Date,,,,,
2026-01-08,77.331778,77.437005,77.293521,77.427444,33516900.0
2026-01-09,77.437007,77.513537,77.437007,77.484840,31849200.0
2026-01-12,77.398731,77.532655,77.389163,77.523087,23983600.0
2026-01-13,77.551783,77.561352,77.456125,77.523087,27607400.0
2026-01-14,77.484822,77.532655,77.417860,77.523087,42994000.0
2026-01-15,77.580480,77.590048,77.484822,77.523087,30412400.0
2026-01-16,77.561377,77.580506,77.475279,77.570938,52466300.0
2026-01-20,77.245678,77.436995,77.169148,77.370033,38653400.0


In [ ]:
# data crosscheck corroborates so we can move on